In [1]:
import torch
torch.randn(6,3)

tensor([[ 0.6941,  0.3407, -1.4234],
        [ 0.8625, -0.1305, -1.0803],
        [-0.8399,  0.3743, -0.1834],
        [ 0.7124, -0.3254, -1.0319],
        [-0.8781, -0.3507,  0.0025],
        [-1.0669, -2.2474, -0.2919]])

In [2]:
# simple example for layer norm
example = torch.randn(2,5)
nn = torch.nn.Sequential(torch.nn.Linear(5,4), torch.nn.ReLU())
out = nn(example)
omean = out.mean(dim=-1, keepdim=True)
ovar = out.var(dim=-1, keepdim=True)
print(out)
print(omean)
print(ovar)
out_norm = (out - omean) / torch.sqrt(ovar)
print(out_norm)

tensor([[0.8140, 0.1152, 0.0000, 0.5465],
        [0.6758, 0.0000, 0.0000, 0.0000]], grad_fn=<ReluBackward0>)
tensor([[0.3689],
        [0.1690]], grad_fn=<MeanBackward1>)
tensor([[0.1434],
        [0.1142]], grad_fn=<VarBackward0>)
tensor([[ 1.1755, -0.6701, -0.9743,  0.4689],
        [ 1.5000, -0.5000, -0.5000, -0.5000]], grad_fn=<DivBackward0>)


In [3]:
from pathlib import Path
import os
os.sys.path.append("../")
from gpt2.dummy_gpt import DummyLayerNorm
layernorm = DummyLayerNorm(5)
out_ln = layernorm(example)


In [4]:
import os

os.sys.path.append("../")

import torch
import torch.nn as nn

from gpt2.dummy_gpt import GELU
from gpt2.config import GPT_CONFIG_124M
# example gelu use
class MyFeedForward(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["embed_dim"], 4 * cfg["embed_dim"]),
            GELU(),
            nn.Linear(4 * cfg["embed_dim"], cfg["embed_dim"])
        )

    def forward(self, x):
        return self.layers(x)

ffn = MyFeedForward(GPT_CONFIG_124M)
x = torch.randn(2,3,768)
y = ffn(x)
print(y.shape)


torch.Size([2, 3, 768])


In [5]:
import sys
sys.path.append("../")
from gpt2.GPT2 import GPTModel
from gpt2.config import GPT_CONFIG_124M
import tiktoken

input shape torch.Size([2, 4, 768])
output shape torch.Size([2, 4, 768])
attn parameters count: 2360064
ff params count: 4722432
torch.Size([2, 4, 50257])
total parameters 163009536
total parameters 85026816
Hello I am confer SH or Hogan salt Invadersity trailersOSE wall


In [6]:
def generate_text(model, idx, max_new_token, context_length):
 
    for _ in range(max_new_token):
        con_idx = idx[:,-context_length:] # idx is batch x token sequence length
        with torch.no_grad():
            logits = model(con_idx)
        # logits is of shape batch x vocab_size
        print(logits.shape)
        logits = logits[:, -1, :]
        probs = torch.softmax(logits, dim=-1)
        tokens = torch.argmax(probs, dim=-1, keepdim=True)
        idx = torch.cat([con_idx, tokens], dim=1)
    return idx

import torch
model = GPTModel(GPT_CONFIG_124M)
model.eval()

tokenizer = tiktoken.get_encoding('gpt2')
text = "Hello I am"
tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = torch.tensor(tokenizer.encode(text)).unsqueeze(0)
generated_text = generate_text(model, encoded_text, 2, 16)
print(generated_text)
tokenizer.decode(generated_text.squeeze(0).tolist())
        



torch.Size([1, 3, 50257])
torch.Size([1, 4, 50257])
tensor([[15496,   314,   716, 19333, 10534]])


'Hello I am Archerateral'

In [7]:
# generrate text

inputs = torch.tensor([[16833, 3626, 6100], 
                       [40, 1107, 588]])

targets = torch.tensor([[3626, 6100, 345], 
                        [1107, 588, 11311]])


with torch.no_grad():
    logits = model(inputs)
probas = torch.softmax(logits, dim=-1)

In [8]:
probas.shape

torch.Size([2, 3, 50257])

In [9]:
target_1 = probas[0,[0,1,2], targets[0]]
target_2 = probas[1,[0,1,2], targets[1] ]
log_probs = torch.log(torch.cat([target_1, target_2]))
avg_log_probs = torch.mean(log_probs)
neg_avg_log_probs = -1 * avg_log_probs
print(neg_avg_log_probs)

tensor(11.0442)


In [10]:
lg = logits.flatten(0,1)
tr = targets.flatten(0,1)
print(lg.shape)
print(tr.shape)

print(torch.nn.functional.cross_entropy(lg,tr))

torch.Size([6, 50257])
torch.Size([6])
tensor(11.0442)


In [11]:
with open('../embeddings/the-verdict.txt', 'r', encoding="utf-8") as f:
    text = f.read()

print(len(text))



20479


In [12]:
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')
print(len(tokenizer.encode(text)))

5145


In [13]:
# lets split train test data
import sys
sys.path.append('../')
n = int(0.9 * len(text))
train = text[0:n]
val = text[n:]

from embeddings.tiktoken_tokenizer import create_dataloader_v2
train_loader = create_dataloader_v2(tokenizer, train, batch_size=16,
                                 max_length=8,
                                 stride=4,
                                 shuffle=True,
                                 drop_last=True,
                                 num_workers=2)

val_loader = create_dataloader_v2(tokenizer, val, batch_size=16, 
                                  max_length=8, 
                                  stride=4, 
                                  shuffle=True,
                                  drop_last=True,
                                  num_workers=2)

INFO:tiktoken:tiktoken version: 0.14.0


In [16]:
#for x,y in train_loader:
#    print(x.shape, y.shape)
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0,1), target_batch.flatten())
    return loss

def calc_loss_loader(dataloader, model, device, num_batches = None):
    total_loss = 0
    if len(dataloader) == 0:
        return float("nan")
    elif num_batches == None:
        num_batches = len(dataloader)
    else:
        num_batches = min(num_batches, len(dataloader))

    for i, (input_batch, target_batch) in enumerate(dataloader):
        if i < num_batches:
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

device = "cpu" 
model.to(device=device)
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)
print(f"train: {train_loss}")
print(f"val: {val_loss}")



INFO:tiktoken:tiktoken version: 0.14.0
INFO:tiktoken:tiktoken version: 0.14.0
INFO:tiktoken:tiktoken version: 0.14.0
INFO:tiktoken:tiktoken version: 0.14.0


train: 10.990308184019277
val: 11.029263615608215


In [ ]:
# main function
def train_simple_model(model, train_loader, batch_loader, optimizer, device,
                       num_epochs, eval_freq, eval_iter, start_context, tokenizer):
    train_losses, val_losses, track_tokens_seen = [], [], []
    tokens_seen, global_step = 0, -1

    for epoch in range(num_epochs):
        model.train()
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()
            loss = calc_loss_batch(input_batch, target_batch, model, device)
            loss.backward()
            optimizer.step()

            tokens_seen += input_batch.numel()
            global_step += 1

            if global_step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(model, train_loader, val_loader, device, eval_iter)
                train_losses.append(train_loss)
                val_losses.append(val_loss)
                track_tokens_seen.append(tokens_seen)
                print(f"Ep {epoch + 1} (Step {global_step:06d}): "
                      f"Train loss: {train_loss:.3f}"
                      f"Val loss: {val_loss:.3f}"
                      )
        generate_and_print_sample(
            model, tokenizer, device, start_context
        )
    return train_losses, val_losses, track_tokens_seen

def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
    
        